# TradeSense · Component 03 · Layer 3 v2 rebuild → `processed_layer3_final_v2`
**Purpose.** A scientifically defensible *synthetic behavioural-scenario* evaluation dataset built on the **genuine Layer 1 Binance BTCUSDT/ETHUSDT 1-minute matrices**, using exactly the frozen six-feature contract. **No model is trained.** Layer 3 v1 and every other dataset are frozen (SHA-256 pinned before, re-checked after).

**What is synthetic, and how.** Layer 1 provides OHLC, volume, executed-trade count, the taker-flow imbalance and the large-fill count. Buy/sell volume is *not stored* in the matrix; it is **derived** from `trade_flow_imbalance` and volume (documented assumption). For every event the notebook draws a *coherent underlying aggregate state* (price displacement path, volume response, taker-share shift, fill-count response, large-fill response, candle-range expansion) from bounded stochastic distributions, applies it **only inside the event window**, then **recomputes the six features from the modified state with the exact Layer 1 formulas**. Genuine bars outside the event's dependency window (event + 15 bars of the volume-acceleration look-back) are bit-identical to Layer 1.

**What it is not.** The labels describe the *injection mechanism*. They are not verified manipulation. Aggregate-level synthesis does **not** claim tick-level physical consistency; spoofing/layering/liquidity-withdrawal are out of scope here (order-book data, TGNN track).

**Scenario families (6 + composites).** A pump-and-dump-like · B fake-breakout-like (reference level from pre-event data only) · C abnormal volume burst · D trade-flow imbalance · E wash-like aggregate proxy · F liquidity-stress proxy · G composites of compatible pairs (all labels recorded; contradictory pairs excluded).

**Isolation.** Events are generated per split inside that split only, with ≥150 bars between events, a 250-bar start margin and a 60-bar end margin; no event or look-back tail crosses a split boundary; independent seeds per symbol/split/event; the test set contains a documented out-of-distribution share.

## 0 · Setup, frozen-input pinning, contract

In [1]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda v: f'{v:,.6g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')

BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L1 = BASE / 'processed_layer1_final_v1'
FROZEN_NAMES = ['processed_layer1_final_v1', 'processed_layer2_final_v3', 'processed_layer2_model_ready_v1', 'processed_layer3_final_v1']
OUT_FINAL = BASE / 'processed_layer3_final_v2'
STAGE = BASE / ('.layer3_v2_stage_' + uuid.uuid4().hex[:8])
assert L1.is_dir(), f'Layer 1 directory missing: {L1}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
GEN_VERSION = 'layer3_v2.0.0'; SRC_VERSION = 'processed_layer1_final_v1'
RUN_STARTED = datetime.now(timezone.utc).isoformat()
print('base:', BASE, '| output (new):', OUT_FINAL.name)

def sha256_file(p, chunk=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def pin_dir(d): return {str(p.relative_to(d)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in sorted(d.rglob('*')) if p.is_file()}
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def auc_rank(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

t0 = time.time()
PIN = {n: pin_dir(BASE / n) for n in FROZEN_NAMES if (BASE / n).is_dir()}
print('pinned frozen directories:', {n: len(v) for n, v in PIN.items()}, f'({time.time()-t0:.0f}s)')
MISSING_FROZEN = [n for n in FROZEN_NAMES if n not in PIN]
if MISSING_FROZEN: print('NOTE: frozen directories not found (nothing to pin):', MISSING_FROZEN)

Mounted at /content/drive
base: /content/drive/MyDrive/research datasets | output (new): processed_layer3_final_v2
pinned frozen directories: {'processed_layer1_final_v1': 400, 'processed_layer2_final_v3': 8, 'processed_layer2_model_ready_v1': 27, 'processed_layer3_final_v1': 5} (50s)


In [2]:
# ---------------- authoritative feature contract (embedded copy of contract v1.0.0, cross-checked below) ----------------
FEATURES = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
SYMS = {'BTCUSDT': 'btc', 'ETHUSDT': 'eth'}
CONTRACT = {'floor': {'BTCUSDT': 2.539920310163499, 'ETHUSDT': 27.765804660846563}, 'thr': {'BTCUSDT': 0.1, 'ETHUSDT': 2.0}}
BOUNDS = {'train': (0, 90720), 'val': (90720, 109440), 'test': (109440, 129600)}; N_BARS = 129600; WARMUP = 15; SPL = ('train', 'val', 'test')
DISCREPANCIES = []

# contract file on Drive (optional) -> integrity + cross-check
cands = [p for d in (BASE, L1, L1 / 'metadata') if d.is_dir() for p in d.glob('feature_contract*.json')]
CONTRACT_FILE = None
for p in cands:
    try:
        c = json.loads(p.read_text())
        ref = {k: v for k, v in c.items() if k != 'integrity'}
        calc = hashlib.sha256(json.dumps(ref, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest()
        ok = c.get('integrity', {}).get('sha256_of_canonical_json_excluding_this_field') == calc
        CONTRACT_FILE = {'path': str(p), 'integrity_ok': bool(ok), 'feature_order_matches': c.get('feature_order') == FEATURES}
        assert c.get('feature_order') == FEATURES, 'contract feature_order differs from the embedded order'
        for s in SYMS:
            pc = c['per_symbol_constants'][s]
            assert math.isclose(pc['volume_acceleration_std_floor_base_units'], CONTRACT['floor'][s], rel_tol=1e-12) and math.isclose(pc['large_trade_threshold_base_units'], CONTRACT['thr'][s], rel_tol=1e-12), 'contract file constants differ'
        break
    except Exception as e:
        DISCREPANCIES.append(f'contract file {p.name} could not be validated: {e}')
print('contract file on Drive:', CONTRACT_FILE or 'not found -> embedded copy of frozen contract v1.0.0 used')

def find_sym_map(obj, hint=None):
    if isinstance(obj, dict):
        if all(k in obj and isinstance(obj[k], (int, float)) for k in SYMS): return {k: float(obj[k]) for k in SYMS}
        for k, v in obj.items():
            if hint is None or re.search(hint, str(k), re.I):
                r = find_sym_map(v)
                if r: return r
        if hint is not None:
            for v in obj.values():
                r = find_sym_map(v)
                if r: return r
    return None
meta = L1 / 'metadata'
thr_file = next(iter(sorted(meta.glob('thresholds*.json'))), None); floor_file = next(iter(sorted(meta.glob('*stabil*.json'))), None)
THR = find_sym_map(json.loads(thr_file.read_text()), 'thresh') if thr_file else None
FLOOR = find_sym_map(json.loads(floor_file.read_text()), 'floor') if floor_file else None
print('Layer 1 metadata -> thresholds:', THR, '| floors:', FLOOR)
for name, got, ref in (('large-trade threshold', THR, CONTRACT['thr']), ('volume-acceleration floor', FLOOR, CONTRACT['floor'])):
    if got is None: DISCREPANCIES.append(f'{name}: Layer 1 metadata file not found/parsable; contract constants used'); continue
    for s in SYMS:
        if not math.isclose(got[s], ref[s], rel_tol=1e-9): DISCREPANCIES.append(f'{name} {s}: metadata {got[s]} != contract {ref[s]} -> METADATA (verified implementation) used')
THR = {s: (THR or CONTRACT['thr'])[s] for s in SYMS}; FLOOR = {s: (FLOOR or CONTRACT['floor'])[s] for s in SYMS}
# documented differences between the task text and the authoritative contract
DISCREPANCIES += ['large_trade_count: task text says fills "exceeding" the threshold; the authoritative contract defines fills with quantity >= threshold. Contract used.',
                  'trade_flow_imbalance: contract formula is (buy-sell)/(buy+sell+1e-8) with 0.0 for empty minutes; buy/sell volume are NOT stored in the Layer 1 matrix, so synthetic buy/sell volume is DERIVED (buy = V*(1+tfi)/2). Raw-tick reconciliation of this feature is still PENDING in the contract ledger.',
                  'warm-up: contract masks bars 0..14 (Layer 1 padded history); Layer 3 convention = volume_acceleration 0.0 for bars 0..14 (a warmup_mask column is provided). Background-parity checks exclude those 15 bars for that one feature.']
for d in DISCREPANCIES: print(' - DISCREPANCY/NOTE:', d)

contract file on Drive: not found -> embedded copy of frozen contract v1.0.0 used
Layer 1 metadata -> thresholds: {'BTCUSDT': 0.1, 'ETHUSDT': 2.0} | floors: {'BTCUSDT': 2.539920310163499, 'ETHUSDT': 27.765804660846563}
 - DISCREPANCY/NOTE: large_trade_count: task text says fills "exceeding" the threshold; the authoritative contract defines fills with quantity >= threshold. Contract used.
 - DISCREPANCY/NOTE: trade_flow_imbalance: contract formula is (buy-sell)/(buy+sell+1e-8) with 0.0 for empty minutes; buy/sell volume are NOT stored in the Layer 1 matrix, so synthetic buy/sell volume is DERIVED (buy = V*(1+tfi)/2). Raw-tick reconciliation of this feature is still PENDING in the contract ledger.
 - DISCREPANCY/NOTE: warm-up: contract masks bars 0..14 (Layer 1 padded history); Layer 3 convention = volume_acceleration 0.0 for bars 0..14 (a warmup_mask column is provided). Background-parity checks exclude those 15 bars for that one feature.


In [3]:
# ---------------- load Layer 1 and PROVE the exact formulas against the stored features ----------------
def vol_acc(vol, floor):
    s = pd.Series(np.asarray(vol, float)); past = s.shift(1).rolling(WARMUP, min_periods=WARMUP)
    va = (s - past.mean()) / np.maximum(past.std(ddof=1), floor)
    out = va.to_numpy(dtype=float).copy(); out[:WARMUP] = 0.0; out[~np.isfinite(out)] = 0.0
    return out

def derived_tfi(buy, sell):
    den = buy + sell
    return np.where(den > 0, (buy - sell) / (den + 1e-8), 0.0)

REQ = ['datetime', 'open', 'high', 'low', 'close', 'volume', 'trades_count'] + FEATURES
ORIG, PART_DT, L1_CHECK = {}, {}, {}
for sym, tag in SYMS.items():
    mp = L1 / 'processed_matrices' / f'{sym.lower()}_l1_feature_matrix_90d.parquet'
    if not mp.is_file():
        alt = sorted((L1 / 'processed_matrices').glob(f'{sym.lower()}*matrix*.parquet')); assert alt, f'matrix for {sym} not found'; mp = alt[0]
    df = pd.read_parquet(mp); miss = [c for c in REQ if c not in df.columns]; assert not miss, f'{sym}: missing columns {miss} (have {list(df.columns)})'
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True); df = df.sort_values('datetime').reset_index(drop=True)
    assert len(df) == N_BARS, f'{sym}: {len(df)} rows != {N_BARS}'
    assert (df['datetime'].diff().dropna() == pd.Timedelta(minutes=1)).all(), f'{sym}: timestamps not a regular 1-minute grid'
    ORIG[sym] = df
    # partitions == slices of the matrix (the verified split definitions)
    for sp, (a, b) in BOUNDS.items():
        names = {'train': ['train'], 'val': ['val', 'validation'], 'test': ['test']}[sp]
        pf = next((q for n in names for q in sorted((L1 / 'partitions').glob(f'{sym.lower()}_{n}_split_*.parquet'))), None)
        assert pf is not None, f'partition file for {sym}/{sp} not found'
        pdx = pd.read_parquet(pf); pdt = pd.to_datetime(pdx['datetime'], utc=True).reset_index(drop=True)
        assert len(pdx) == b - a and (pdt == df['datetime'].iloc[a:b].reset_index(drop=True)).all(), f'{sym}/{sp}: partition != matrix slice'
        assert all(np.array_equal(pdx[f].to_numpy(), df[f].iloc[a:b].to_numpy()) for f in FEATURES), f'{sym}/{sp}: partition features != matrix features'
    # exact feature recomputation from the stored raw columns
    c = df['close'].to_numpy(float); lr = np.r_[0.0, np.log(c[1:] / c[:-1])]
    hl = (df['high'].to_numpy(float) - df['low'].to_numpy(float)) / c; rush = df['trades_count'].to_numpy(float) / (df['volume'].to_numpy(float) + 1e-8)
    va = vol_acc(df['volume'].to_numpy(float), FLOOR[sym]); st = df['volume_acceleration'].to_numpy(float)
    rel = np.abs(va[WARMUP:] - st[WARMUP:]) / np.maximum(np.abs(st[WARMUP:]), 1e-12)
    chk = {'log_returns_max_abs_err': float(np.abs(lr - df['log_returns'].to_numpy(float)).max()), 'hl_max_abs_err': float(np.abs(hl - df['hl_spread_ratio'].to_numpy(float)).max()),
           'rush_max_rel_err': float((np.abs(rush - df['rush_order_density'].to_numpy(float)) / np.maximum(np.abs(df['rush_order_density'].to_numpy(float)), 1e-12)).max()),
           'volume_acceleration_max_rel_err_after_warmup': float(rel.max()), 'volume_acceleration_p99_rel_err': float(np.quantile(rel, .99)),
           'tfi_in_[-1,1]': bool(df['trade_flow_imbalance'].between(-1, 1).all()), 'large_is_nonneg_integer_valued': bool((df['large_trade_count'] >= 0).all() and np.allclose(df['large_trade_count'], np.round(df['large_trade_count']))),
           'large_le_trades': bool((df['large_trade_count'] <= df['trades_count']).all()),
           'share_large_x_thr_le_volume': float((df['large_trade_count'] * THR[sym] <= df['volume'] * (1 + 1e-9)).mean()),
           'nonpositive_prices': int((df[['open', 'high', 'low', 'close']] <= 0).any(axis=1).sum()), 'ohlc_violations': int(((df['high'] < df[['open', 'close']].max(axis=1) - 1e-12) | (df['low'] > df[['open', 'close']].min(axis=1) + 1e-12)).sum()),
           'dtypes': {c_: str(df[c_].dtype) for c_ in FEATURES}}
    assert chk['log_returns_max_abs_err'] < 1e-12 and chk['hl_max_abs_err'] < 1e-12 and chk['rush_max_rel_err'] < 1e-9, f'{sym}: Layer 1 formulas not reproduced {chk}'
    assert chk['volume_acceleration_max_rel_err_after_warmup'] < 1e-6, f'{sym}: volume_acceleration NOT reproduced with the verified floor {chk}'
    L1_CHECK[sym] = chk
    print(sym, json.dumps({k: v for k, v in chk.items() if k != 'dtypes'}, default=jsonable))
FEATURE_DTYPES = {s: L1_CHECK[s]['dtypes'] for s in SYMS}
print('Layer 1 verified: formulas reproduced from stored raw columns; partitions equal matrix slices; BTC/ETH on the same minute grid:', bool((ORIG['BTCUSDT'].datetime == ORIG['ETHUSDT'].datetime).all()))

BTCUSDT {"log_returns_max_abs_err": 0.0, "hl_max_abs_err": 0.0, "rush_max_rel_err": 0.0, "volume_acceleration_max_rel_err_after_warmup": 1.666224457626611e-10, "volume_acceleration_p99_rel_err": 9.51362429888518e-12, "tfi_in_[-1,1]": true, "large_is_nonneg_integer_valued": true, "large_le_trades": true, "share_large_x_thr_le_volume": 1.0, "nonpositive_prices": 0, "ohlc_violations": 0}
ETHUSDT {"log_returns_max_abs_err": 0.0, "hl_max_abs_err": 0.0, "rush_max_rel_err": 0.0, "volume_acceleration_max_rel_err_after_warmup": 2.557953845547366e-08, "volume_acceleration_p99_rel_err": 1.9152666088660456e-11, "tfi_in_[-1,1]": true, "large_is_nonneg_integer_valued": true, "large_le_trades": true, "share_large_x_thr_le_volume": 1.0, "nonpositive_prices": 0, "ohlc_violations": 0}
Layer 1 verified: formulas reproduced from stored raw columns; partitions equal matrix slices; BTC/ETH on the same minute grid: True


## 1 · Pre-registered generation configuration and scenario machinery
All distributions are bounded and stochastic; parameters are drawn **per event** from the event's own seed, so one event can be reproduced without the others. Distribution families are fixed here, **before** any model exists.

In [4]:
CFG = {'generation_version': GEN_VERSION, 'source_dataset_version': SRC_VERSION, 'master_seed': 20261011,
       'events_per_symbol_split': {'train': 80, 'val': 30, 'test': 36}, 'composite_share': 0.10, 'ood_share_test': 0.25,
       'severity_probs': {'subtle': 0.35, 'medium': 0.40, 'severe': 0.25},
       'tail_bars': WARMUP, 'min_gap_bars': 150, 'pad_start_bars': 250, 'pad_end_bars': 60, 'sigma_lookback': 240, 'max_abs_log_displacement': 0.08,
       'duration_range_bars': {'A': [15, 90], 'B': [10, 45], 'C': [5, 40], 'D': [10, 80], 'E': [15, 90], 'F': [10, 90]}, 'ood_duration_multiplier': [1.5, 2.2], 'ood_duration_cap': 160,
       'severity_ranges': {'subtle': {'a': [0.4, 1.0], 'M': [1.2, 2.0], 'f': [0.05, 0.12], 'R': [1.0, 1.4], 'tb': 0.5},
                           'medium': {'a': [1.0, 2.2], 'M': [2.0, 4.5], 'f': [0.12, 0.25], 'R': [1.4, 2.5], 'tb': 1.5},
                           'severe': {'a': [2.2, 4.5], 'M': [4.5, 12.0], 'f': [0.25, 0.42], 'R': [2.5, 5.0], 'tb': 3.0}},
       'ood_scale': {'a': 1.4, 'M': 1.4, 'f': 1.2, 'R': 1.4, 'tb': 1.3}, 'weak_volume_response_prob': 0.15, 'fingerprint_min_distance': 1e-6,
       'trivial_signature_single_feature_auc': 0.95}
FAMS = {'A': 'pump_and_dump_like', 'B': 'fake_breakout_like', 'C': 'abnormal_volume_burst', 'D': 'trade_flow_imbalance', 'E': 'wash_like_aggregate_proxy', 'F': 'liquidity_stress_proxy'}
LABEL_COLS = ['label_' + FAMS[k] for k in 'ABCDEF']
COMPAT = [('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('D', 'C'), ('F', 'C'), ('F', 'D')]     # (price-driving primary, non-price secondary); contradictory pairs (e.g. wash-like + directional) excluded
TAIL, MIN_GAP, PAD_S, PAD_E = CFG['tail_bars'], CFG['min_gap_bars'], CFG['pad_start_bars'], CFG['pad_end_bars']
PHASES = {0: 'none', 1: 'initiation', 2: 'escalation', 3: 'peak', 4: 'reversal_recovery'}

def U(rng, lo, hi): return float(rng.uniform(lo, hi))
def LU(rng, lo, hi): return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
def ar1(rng, n, phi, sd):
    e = rng.normal(0, sd, n); x = np.zeros(n)
    for i in range(n): x[i] = (phi * x[i - 1] if i else 0.0) + e[i]
    return x
def smooth(x, k=3): return np.convolve(x, np.ones(k) / k, mode='same') if len(x) >= k else x
def bump(u, p, a, b): return np.where(u <= p, (u / p) ** a, ((1 - u) / (1 - p)) ** b)
def shape_profile(u, sh):
    g = bump(u, sh['p'], sh['a'], sh['b'])
    if sh['kind'] == 'plateau': g = np.minimum(1.0, 2.5 * g)
    elif sh['kind'] == 'double_peak': g = g + sh['w2'] * bump(u, sh['p2'], sh['a'], sh['b'])
    m = g.max(); return g / m if m > 0 else g
def draw_shape(rng, ood):
    if ood:     # out-of-distribution region: late peaks, steep rises, slow recovery (disjoint from the training ranges)
        kind = str(rng.choice(['ramp_decay', 'double_peak', 'plateau'])); sh = {'kind': kind, 'p': U(rng, 0.75, 0.92), 'a': U(rng, 2.5, 4.5), 'b': U(rng, 0.3, 0.6)}
    else:
        kind = str(rng.choice(['ramp_decay', 'spike_decay', 'plateau', 'double_peak']))
        sh = ({'kind': kind, 'p': U(rng, 0.08, 0.25), 'a': U(rng, 0.4, 1.2), 'b': U(rng, 1.5, 4.0)} if kind == 'spike_decay' else {'kind': kind, 'p': U(rng, 0.2, 0.7), 'a': U(rng, 0.7, 3.0), 'b': U(rng, 0.7, 3.0)})
    sh['p2'] = float(min(0.95, sh['p'] + U(rng, 0.2, 0.4))); sh['w2'] = U(rng, 0.4, 0.8); sh['ood'] = bool(ood); return sh
def sev_ranges(sev, ood):
    S = {k: (list(v) if isinstance(v, list) else v) for k, v in CFG['severity_ranges'][sev].items()}
    if ood:
        sc = CFG['ood_scale']
        for k in ('a', 'M', 'R'): S[k] = [S[k][0], S[k][1] * sc[k]]
        S['f'] = [S['f'][0], min(0.48, S['f'][1] * sc['f'])]; S['tb'] = S['tb'] * sc['tb']
    return S
def mom_from_path(d):
    m = np.diff(d, prepend=0.0); s = np.abs(m).max(); return m / s if s > 0 else m

def nonprice(L, S, ctx, rng, ood, spec, mom):
    uv = (np.arange(L) + 0.5) / L; shv = draw_shape(rng, ood)
    if spec.get('vol_peak_u') is not None: shv['p'] = float(np.clip(spec['vol_peak_u'] + U(rng, -0.1, 0.1), 0.05, 0.95))
    gv = shape_profile(uv, shv); logM = math.log(LU(rng, *S['M'])) * spec['vol_scale']
    weak = bool(rng.random() < max(CFG['weak_volume_response_prob'] * spec.get('weak_mult', 1.0), 0.0)); logM = logM * 0.15 if weak else logM
    vmult = np.exp(logM * gv + rng.normal(0, 0.2, L)); famp = U(rng, *S['f']) * spec.get('flow_scale', 1.0); mode = spec['flow_mode']; gf = None; sgn = int(rng.choice([-1, 1]))
    if mom is None: mom = smooth(mom_from_path(gv))
    if mode == 'mom': pflow = famp * smooth(mom)
    elif mode == 'persist':
        shf = draw_shape(rng, ood); gf = shape_profile(uv, shf); pflow = sgn * famp * gf
    elif mode == 'noise': pflow = 0.3 * famp * sgn * gv + np.cumsum(rng.normal(0, 0.03, L)) * (4 * uv * (1 - uv))
    else: pflow = np.zeros(L)
    pflow = pflow + rng.normal(0, 0.03, L)
    pull = np.clip(spec.get('pull_amp', 0.0) * gv, 0.0, 0.97)
    beta = U(rng, *spec['beta']); gamma = U(rng, *spec['gamma']); tb = S['tb'] * spec.get('tb_scale', 0.0) * gv
    tmult = vmult ** beta * (1 + tb) * np.exp(rng.normal(0, 0.15, L)); lmult = vmult ** gamma * np.exp(rng.normal(0, 0.3, L))
    lrate = U(rng, 0.0, 0.6) * ctx['large_mean'] * np.maximum(vmult - 1, 0); R = LU(rng, *S['R']) * spec.get('R_scale', 1.0)
    rmult = np.maximum(1.0, 1 + (R - 1) * gv * np.exp(rng.normal(0, 0.2, L)))
    pk = {'vol_shape': shv['kind'], 'vol_p': shv['p'], 'peak_vmult_target': math.exp(logM), 'weak_volume_response': weak, 'flow_mode': mode, 'flow_amp': famp, 'beta': beta, 'gamma': gamma, 'R': R, 'pull_amp': spec.get('pull_amp', 0.0)}
    return {'vmult': np.clip(vmult, 0.2, 30.0), 'pflow': pflow, 'pull': pull, 'tmult': tmult, 'lmult': lmult, 'lrate': lrate, 'rmult': rmult, 'u_up': rng.uniform(0, 1, L), 'u_dn': rng.uniform(0, 1, L), 'env': gv, 'gf': gf, 'sgn': sgn, 'params': pk}

def price_bump(L, rng, ood, sig, a_rng, S, sign, scale=1.0, noise_sd=0.35, undershoot=False):
    up = np.arange(1, L + 1) / L; sh = draw_shape(rng, ood); g = shape_profile(up, sh)
    A = min(LU(rng, *a_rng) * sig * math.sqrt(L) * scale, CFG['max_abs_log_displacement'])
    ud = U(rng, 0, 0.4) if (undershoot and rng.random() < 0.5) else 0.0
    h = np.where(up > sh['p'], np.sin(np.pi * (up - sh['p']) / (1 - sh['p'] + 1e-12)), 0.0)
    d = sign * A * (g - ud * h) + ar1(rng, L, 0.6, noise_sd * sig) * (4 * up * (1 - up)); d[-1] = 0.0
    return d, {'price_shape': sh['kind'], 'price_p': sh['p'], 'price_a': sh['a'], 'price_b': sh['b'], 'amp_log': A, 'undershoot': ud, 'sign': sign}, up[int(np.argmax(np.abs(d)))]

def eff_generic(fam, L, sev, ood, ctx, rng, primary):
    S = sev_ranges(sev, ood); sig = ctx['sigma']; d = np.zeros(L); pp = {}; peak_u = None; mom = None
    if fam == 'A':
        if primary: d, pp, peak_u = price_bump(L, rng, ood, sig, S['a'], S, +1, undershoot=True); mom = mom_from_path(d)
        spec = dict(vol_scale=1.0, flow_mode='mom', beta=(0.5, 1.0), gamma=(0.8, 1.5), vol_peak_u=peak_u)
    elif fam == 'C':
        if primary: d, pp, peak_u = price_bump(L, rng, ood, sig, (0.1, 0.5), S, int(rng.choice([-1, 1])), noise_sd=0.2)
        spec = dict(vol_scale=1.3, weak_mult=0.0, flow_mode='noise', beta=(0.7, 1.1), gamma=(0.6, 1.4), R_scale=0.5)
    elif fam == 'D':
        spec = dict(vol_scale=0.5, weak_mult=1.3, flow_mode='persist', beta=(0.4, 0.9), gamma=(0.6, 1.2), R_scale=0.5)
    elif fam == 'E':
        if primary: d, pp, peak_u = price_bump(L, rng, ood, sig, (0.05, 0.3), S, int(rng.choice([-1, 1])), noise_sd=0.2)
        spec = dict(vol_scale=0.8, weak_mult=0.7, flow_mode='pull', beta=(0.9, 1.2), gamma=(0.2, 0.8), pull_amp=U(rng, 0.4, 0.95), tb_scale=U(rng, 0.7, 1.3), R_scale=0.3)
    elif fam == 'F':
        if primary: d, pp, peak_u = price_bump(L, rng, ood, sig, S['a'], S, int(rng.choice([-1, 1])), noise_sd=0.8)
        spec = dict(vol_scale=0.7, weak_mult=1.3, flow_mode='noise', beta=(0.5, 1.0), gamma=(0.5, 1.2), R_scale=1.3)
    npk = nonprice(L, S, ctx, rng, ood, spec, mom)
    if fam == 'D' and primary:       # persistent one-sided pressure with partial price impact that fully relaxes by the end of the event
        up = np.arange(1, L + 1) / L; uv = (np.arange(L) + 0.5) / L; gfp = shape_profile(up, draw_shape(rng, ood))
        A = min(LU(rng, *S['a']) * sig * math.sqrt(L) * 0.6, CFG['max_abs_log_displacement']) * U(rng, 0.2, 0.7)
        d = npk['sgn'] * A * gfp + ar1(rng, L, 0.6, 0.2 * sig) * (4 * up * (1 - up)); d[-1] = 0.0; pp = {'price_shape': 'follows_flow', 'amp_log': A, 'sign': npk['sgn']}
    env = npk['env'] if not np.any(d) else np.maximum(npk['env'], np.abs(d) / max(np.abs(d).max(), 1e-12))
    return d, npk, env, {**pp, **npk['params']}

def eff_B(L, sev, ood, ctx, rng, primary):
    S = sev_ranges(sev, ood); sig = ctx['sigma']
    if not primary:
        npk = nonprice(L, S, ctx, rng, ood, dict(vol_scale=0.9, flow_mode='mom', beta=(0.5, 1.0), gamma=(0.6, 1.3)), None); return np.zeros(L), npk, npk['env'], npk['params']
    up = np.arange(1, L + 1) / L; sh = draw_shape(rng, ood); g = shape_profile(up, sh); noise = ar1(rng, L, 0.6, 0.08) * (4 * up * (1 - up)); dshape = g + noise; dshape[-1] = 0.0
    lnC = np.log(ctx['C_win']); Hh, Ll = ctx['H_hist'], ctx['L_hist']; kmax = min(CFG['max_abs_log_displacement'], ctx['step_room'] / max(float(np.abs(np.diff(dshape, prepend=0.0)).max()), 1e-12)); over = LU(rng, *S['a']) * 0.5 * sig
    Wlist = [int(rng.integers(10, 121)), 120, 60, 30, 15, 8]; best = None
    for W in Wlist:
        cand = []
        for side in (+1, -1):
            ref = float(Hh[-W:].max()) if side == 1 else float(Ll[-W:].min()); target = math.log(ref) + side * over
            f = lambda k: (np.max(side * (lnC + k * dshape * side)) - side * target)   # side*d*side keeps the excursion in the breakout direction
            if f(0.0) >= 0: k = 1e-6
            elif f(kmax) < 0: continue
            else:
                lo, hi = 0.0, kmax
                for _ in range(50):
                    mid = 0.5 * (lo + hi)
                    if f(mid) >= 0: hi = mid
                    else: lo = mid
                k = hi
            ce = ctx['C_win'][-1]; inside = bool(Ll[-W:].min() <= ce <= ref) if side == 1 else bool(ref <= ce <= Hh[-W:].max()); cand.append((k, side, ref, W, inside))
        if cand:
            ok = [c for c in cand if c[4]]
            if ok:
                ok.sort(key=lambda c: c[0]); best = ok[0] if rng.random() < 0.7 or len(ok) == 1 else ok[-1]; break
            if best is None: best = sorted(cand, key=lambda c: c[0])[0]
    if best is None:                       # extreme-context fallback: tiny excursion, flagged
        best = (kmax, int(rng.choice([-1, 1])), float(Hh[-8:].max()), 8, False)
    k, side, ref, W, inside = best; d = k * side * dshape; d[-1] = 0.0
    peak_u = up[int(np.argmax(side * d))]; npk = nonprice(L, S, ctx, rng, ood, dict(vol_scale=0.9, flow_mode='mom', beta=(0.5, 1.0), gamma=(0.6, 1.3), vol_peak_u=peak_u), mom_from_path(d))
    brk = int(np.sum(side * (lnC + d) > side * math.log(ref)))
    env = np.maximum(npk['env'], np.abs(d) / max(np.abs(d).max(), 1e-12))
    return d, npk, env, {'price_shape': sh['kind'], 'price_p': sh['p'], 'price_a': sh['a'], 'price_b': sh['b'], 'amp_log': k, 'sign': side, 'ref_window_bars': W, 'ref_level': ref, 'bars_closing_beyond_ref': brk, 'returns_inside_range': inside, **npk['params']}

def build_pack(fams, L, sev, ood, ctx, rng):
    comps, prim = [], True
    for fam in fams:
        d, npk, env, pp = (eff_B if fam == 'B' else (lambda *a: eff_generic(fam, *a)))(L, sev, ood, ctx, rng, prim)
        comps.append({'fam': fam, 'd': d, 'npk': npk, 'env': env, 'params': pp}); prim = False
    d = comps[0]['d'].copy(); d[-1] = 0.0; vm, pf, pu, tm, lm, lr_, rm = [np.ones(L), np.zeros(L), np.zeros(L), np.ones(L), np.ones(L), np.zeros(L), np.ones(L)]
    env = np.zeros(L)
    for c in comps:
        n = c['npk']; vm = vm * n['vmult']; pf = pf + n['pflow']; pu = np.maximum(pu, n['pull']); tm = tm * n['tmult']; lm = lm * n['lmult']; lr_ = lr_ + n['lrate']; rm = np.maximum(rm, n['rmult']); env = np.maximum(env, c['env'])
    return {'d': d, 'vmult': np.clip(vm, 0.2, 30.0), 'pflow': np.clip(pf, -0.6, 0.6), 'pull': pu, 'tmult': tm, 'lmult': lm, 'lrate': lr_, 'rmult': rm, 'u_up': comps[0]['npk']['u_up'], 'u_dn': comps[0]['npk']['u_dn'], 'env': env,
            'components': [{'family': c['fam'], 'params': c['params']} for c in comps]}
print('scenario machinery defined; compatible composite pairs:', COMPAT)

scenario machinery defined; compatible composite pairs: [('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('D', 'C'), ('F', 'C'), ('F', 'D')]


## 2 · Per-symbol generation: placement → state modification → exact feature recomputation
Placement is done **inside each split** with hard margins. Context quantities (local σ, reference levels, large-fill level) come from the **original** Layer 1 series *before* the event start only.

In [5]:
def draw_specs(split, n_events, rng):
    n_comp = int(round(CFG['composite_share'] * n_events)); n_single = n_events - n_comp
    order = list('ABCDEF'); singles = [order[i % 6] for i in range(n_single)]
    fam_lists = [[f] for f in singles] + [list(COMPAT[int(rng.integers(0, len(COMPAT)))]) for _ in range(n_comp)]
    rng.shuffle(fam_lists)
    sev_names = list(CFG['severity_probs']); sevs = list(rng.choice(sev_names, size=n_events, p=[CFG['severity_probs'][s] for s in sev_names]))
    ood = np.zeros(n_events, bool)
    if split == 'test': ood[rng.choice(n_events, size=int(round(CFG['ood_share_test'] * n_events)), replace=False)] = True
    specs = []
    for i in range(n_events):
        fams = fam_lists[i]; lo, hi = CFG['duration_range_bars'][fams[0]]; L = LU(rng, lo, hi)
        if ood[i]: L = min(L * U(rng, *CFG['ood_duration_multiplier']), CFG['ood_duration_cap'])
        specs.append({'families': fams, 'severity': str(sevs[i]), 'ood': bool(ood[i]), 'L': int(round(L)), 'seed': int(rng.integers(1, 2 ** 31 - 1))})
    return specs

def place(specs, a, b, rng):
    N = len(specs); dropped = 0
    def slack_of(sp_): return (b - 1 - PAD_E) - (a + PAD_S) + 1 - sum(s['L'] + TAIL for s in sp_) - (len(sp_) - 1) * (MIN_GAP + 1)
    while N and slack_of(specs[:N]) < 0: N -= 1; dropped += 1
    specs = specs[:N]; slack = slack_of(specs); cuts = np.sort(rng.integers(0, slack + 1, N)) if N else np.array([], int)
    extras = np.diff(np.r_[0, cuts, slack]) if N else []
    s = a + PAD_S + (extras[0] if N else 0); out = []
    for i, sp_ in enumerate(specs):
        if i: s = out[-1]['end'] + TAIL + 1 + MIN_GAP + extras[i]
        sp_ = dict(sp_); sp_['start'] = int(s); sp_['end'] = int(s + sp_['L'] - 1); out.append(sp_)
    return out, dropped

def generate_symbol(sym_i, sym):
    o = ORIG[sym]; O = o['open'].to_numpy(float).copy(); H = o['high'].to_numpy(float).copy(); Lw = o['low'].to_numpy(float).copy(); C = o['close'].to_numpy(float).copy()
    V = o['volume'].to_numpy(float).copy(); TR = o['trades_count'].to_numpy(np.int64).copy(); LG = np.rint(o['large_trade_count'].to_numpy(float)).astype(np.int64)
    TFI0 = o['trade_flow_imbalance'].to_numpy(float); BUY = V * (1 + TFI0) / 2; SELL = V - BUY
    O0, H0, L0, C0, V0, TR0, LG0 = O.copy(), H.copy(), Lw.copy(), C.copy(), V.copy(), TR.copy(), LG.copy()
    F = {f: o[f].to_numpy(float).copy() for f in FEATURES}
    lr0 = o['log_returns'].to_numpy(float); tr_end = BOUNDS['train'][1]; LR_CAP = 1.5 * float(np.abs(lr0[:tr_end]).max()); HL_CAP = 1.5 * float(((H0 - L0) / C0)[:tr_end].max()); sig_roll = pd.Series(lr0).rolling(CFG['sigma_lookback'], min_periods=CFG['sigma_lookback']).std(ddof=1).to_numpy()
    vol_roll = pd.Series(V0).rolling(60, min_periods=60).mean().to_numpy(); sig_pct = pd.Series(sig_roll).rank(pct=True).to_numpy(); vol_pct = pd.Series(vol_roll).rank(pct=True).to_numpy()
    n = len(o); label = np.zeros(n, np.int8); dep = np.zeros(n, np.int8); phase = np.zeros(n, np.int8); inten = np.zeros(n, np.float32); ood_b = np.zeros(n, np.int8); sev_c = np.zeros(n, np.int8)
    ev_id = np.full(n, '', dtype=object); dep_id = np.full(n, '', dtype=object); sc_type = np.full(n, 'normal', dtype=object); sc_fam = np.full(n, 'normal', dtype=object)
    fam_lab = {c: np.zeros(n, np.int8) for c in LABEL_COLS}; events = []; dropped = {}
    for si, sp in enumerate(SPL):
        a, b = BOUNDS[sp]; rng = np.random.default_rng(np.random.SeedSequence([CFG['master_seed'], sym_i, si]))
        specs = draw_specs(sp, CFG['events_per_symbol_split'][sp], rng); placed, nd = place(specs, a, b, rng); dropped[sp] = nd
        for k, ev in enumerate(placed):
            L = ev['L']; lo_s = (placed[k - 1]['dep_end'] + MIN_GAP + 1) if k else (a + PAD_S)
            hi_s = (placed[k + 1]['start'] - MIN_GAP - 1 - TAIL - L + 1) if k + 1 < len(placed) else (b - 1 - PAD_E - TAIL - L + 1)
            offsets = [0] + ([5, -5, 10, -10, 20, -20, 40, -40, 80, -80] if ev['families'][0] == 'B' else []); chosen = None
            for off in offsets:           # fake-breakout events may slide inside their free gap until the original price ends back inside the earlier range
                s = ev['start'] + off; e = s + L - 1
                if s < lo_s or s > hi_s: continue
                ctx = {'sigma': max(float(sig_roll[s - 1]), 1e-5), 'large_mean': float(LG0[s - 60:s].mean()) + 0.5, 'H_hist': H0[s - 120:s], 'L_hist': L0[s - 120:s], 'C_win': C0[s:e + 1],
                       'step_room': max(LR_CAP - float(np.abs(lr0[s:e + 1]).max()), 0.2 * LR_CAP)}
                rng_e = np.random.default_rng(ev['seed']); pk = build_pack(ev['families'], L, ev['severity'], ev['ood'], ctx, rng_e)
                okB = (ev['families'][0] != 'B') or bool(pk['components'][0]['params'].get('returns_inside_range', False)) and pk['components'][0]['params'].get('bars_closing_beyond_ref', 0) >= 1
                if chosen is None: chosen = (s, e, ctx, pk, rng_e)
                if okB: chosen = (s, e, ctx, pk, rng_e); break
            s, e, ctx, pk, rng_e = chosen; ev['start'], ev['end'], ev['dep_end'] = s, e, e + TAIL; d = pk['d']; sl = slice(s, e + 1); cap_scale = 1.0
            if ev['families'][0] != 'B':       # plausibility: no single-minute move above 1.5x the largest TRAIN-period real move (shape kept, amplitude reduced)
                ms = float(np.abs(np.diff(np.r_[0.0, d])).max()); room = ctx['step_room']
                if ms > room: cap_scale = room / ms; d = d * cap_scale; d[-1] = 0.0; pk['d'] = d
            dprev = np.r_[0.0, d[:-1]]; Cn = C0[sl] * np.exp(d); On = O0[sl] * np.exp(dprev); rg = H0[sl] - L0[sl]
            Hn = H0[sl] * np.exp(np.maximum(dprev, d)) + (pk['rmult'] - 1) * rg * pk['u_up']; Ln = np.maximum(L0[sl] * np.exp(np.minimum(dprev, d)) - (pk['rmult'] - 1) * rg * pk['u_dn'], 0.5 * L0[sl] * np.exp(np.minimum(dprev, d)))
            hln = (Hn - Ln) / Cn; over = np.maximum(hln - HL_CAP, 0.0) * Cn
            if over.any(): Hn = np.maximum(Hn - over / 2, np.maximum(On, Cn)); Ln = np.minimum(Ln + over / 2, np.minimum(On, Cn))
            Vn = V0[sl] * pk['vmult']; p0 = (1 + TFI0[sl]) / 2; pn = np.clip(p0 + pk['pflow'], 0.01, 0.99); pn = pn + pk['pull'] * (0.5 - pn)
            bn, sn = Vn * pn, Vn * (1 - pn); trn = np.where(Vn > 0, np.maximum(1, np.rint(TR0[sl] * pk['tmult'])), 0).astype(np.int64)
            lgn = np.rint(LG0[sl] * pk['lmult']) + rng_e.poisson(pk['lrate']); lgn = np.minimum(np.minimum(lgn, trn), np.floor(Vn / THR[sym] + 1e-12)).astype(np.int64); lgn = np.maximum(lgn, 0)
            O[sl], H[sl], Lw[sl], C[sl], V[sl], TR[sl], LG[sl], BUY[sl], SELL[sl] = On, Hn, Ln, Cn, Vn, trn, lgn, bn, sn
            prevC = np.r_[C0[s - 1], Cn[:-1]]; F['log_returns'][sl] = np.log(Cn / prevC); F['trade_flow_imbalance'][sl] = derived_tfi(bn, sn)
            F['hl_spread_ratio'][sl] = (Hn - Ln) / Cn; F['rush_order_density'][sl] = trn / (Vn + 1e-8); F['large_trade_count'][sl] = lgn
            env = pk['env']; ip = int(np.argmax(env)); half = max(1, int(round(0.1 * L))); idx = np.arange(L)
            ph = np.where(idx < ip, np.where(idx < ip / 2, 1, 2), 4); ph[np.abs(idx - ip) <= half] = 3
            eid = f'{sym}-{sp}-{k:03d}'; fams = ev['families']; ftxt = '+'.join(FAMS[f] for f in fams)
            label[sl] = 1; phase[sl] = ph; inten[sl] = env.astype(np.float32); ood_b[sl] = int(ev['ood']); sev_c[sl] = {'subtle': 1, 'medium': 2, 'severe': 3}[ev['severity']]
            ev_id[sl] = eid; sc_type[sl] = ('composite:' + ftxt) if len(fams) > 1 else FAMS[fams[0]]; sc_fam[sl] = 'composite' if len(fams) > 1 else FAMS[fams[0]]
            for f in fams: fam_lab['label_' + FAMS[f]][sl] = 1
            dep[e + 1:e + TAIL + 1] = 1; dep_id[e + 1:e + TAIL + 1] = eid
            prm = {'families': fams, 'severity': ev['severity'], 'ood': ev['ood'], 'duration': L, 'components': pk['components']}
            ph_hash = hashlib.sha1(json.dumps(prm, sort_keys=True, default=lambda x: round(float(x), 6) if isinstance(x, (np.floating, float)) else str(x)).encode()).hexdigest()
            events.append({'event_id': eid, 'symbol': sym, 'split': sp, 'scenario_type': sc_type[s], 'scenario_family': sc_fam[s], 'families_json': json.dumps(fams), 'composite': int(len(fams) > 1),
                           'start_idx': s, 'end_idx': e, 'dependency_end_idx': e + TAIL, 'start_timestamp': o['datetime'].iloc[s], 'end_timestamp': o['datetime'].iloc[e], 'duration_bars': L, 'severity': ev['severity'], 'is_ood': int(ev['ood']),
                           'random_seed': ev['seed'], 'generation_version': GEN_VERSION, 'source_dataset_version': SRC_VERSION, 'sigma_local': ctx['sigma'], 'sigma_pct': float(sig_pct[s - 1]), 'volume_pct': float(vol_pct[s - 1]),
                           'speed_cap_scale': float(cap_scale), 'peak_abs_log_displacement': float(np.abs(d).max()), 'peak_volume_multiplier': float(pk['vmult'].max()), 'peak_flow_shift': float(np.abs(pk['pflow']).max()), 'peak_range_multiplier': float(pk['rmult'].max()),
                           'peak_trade_multiplier': float(pk['tmult'].max()), 'price_shape': next((c['params'].get('price_shape') for c in pk['components'] if c['params'].get('price_shape')), 'none'),
                           'vol_shape': pk['components'][0]['params'].get('vol_shape'), 'params_hash': ph_hash, 'params_json': json.dumps(prm, default=lambda x: round(float(x), 8) if isinstance(x, (np.floating, float)) else str(x))})
    va = vol_acc(V, FLOOR[sym]); dep_mask = np.zeros(n, bool)
    for ev in events: dep_mask[ev['start_idx']:ev['dependency_end_idx'] + 1] = True
    F['volume_acceleration'][dep_mask] = va[dep_mask]; F['volume_acceleration'][:WARMUP] = 0.0
    normal = (~dep_mask).astype(np.int8)
    state = pd.DataFrame({'datetime': o['datetime'], 'open': O, 'high': H, 'low': Lw, 'close': C, 'volume': V, 'buy_volume_derived': BUY, 'sell_volume_derived': SELL, 'trades_count': TR, 'large_trade_count': LG})
    feats = pd.DataFrame({'datetime': o['datetime'], **{f: F[f].astype(FEATURE_DTYPES[sym][f]) if FEATURE_DTYPES[sym][f] != 'object' else F[f] for f in FEATURES}})
    labels = pd.DataFrame({'datetime': o['datetime'], 'synthetic_label': label, 'normal_background': normal, 'dependency_tail': dep, 'warmup_mask': (np.arange(n) < WARMUP).astype(np.int8), 'event_id': ev_id.astype(str), 'dependency_event_id': dep_id.astype(str),
                           'scenario_type': sc_type.astype(str), 'scenario_family': sc_fam.astype(str), 'severity_code': sev_c, 'is_ood': ood_b, 'phase_code': phase, 'intensity': inten, **{c: fam_lab[c] for c in LABEL_COLS}})
    return {'state': state, 'features': feats, 'labels': labels, 'events': pd.DataFrame(events), 'dropped': dropped}

def gen_hash(res):
    h = hashlib.sha256()
    for k in ('state', 'features', 'labels'):
        for c in res[k].columns:
            v = res[k][c]; h.update(c.encode())
            if v.dtype.kind in 'biuf': h.update(np.ascontiguousarray(v.to_numpy()).tobytes())
            elif v.dtype.kind == 'M': h.update(np.ascontiguousarray(v.astype('int64').to_numpy()).tobytes())
            else: h.update('\x1f'.join(v.astype(str).tolist()).encode('utf-8'))
    h.update(res['events'].drop(columns=['start_timestamp', 'end_timestamp']).astype(str).to_csv(index=False).encode()); return h.hexdigest()
print('generator defined')

generator defined


In [6]:
t0 = time.time()
GEN = {sym: generate_symbol(i, sym) for i, sym in enumerate(SYMS)}
GEN_HASH = {sym: gen_hash(GEN[sym]) for sym in SYMS}
print(f'generated in {time.time()-t0:.0f}s | hashes:', {k: v[:16] for k, v in GEN_HASH.items()})
EV = pd.concat([GEN[s]['events'] for s in SYMS], ignore_index=True)
display(EV.groupby(['symbol', 'split']).size().unstack().T)
display(pd.crosstab([EV.symbol, EV.split], EV.scenario_family))
print('events dropped by the placement budget:', {s: GEN[s]['dropped'] for s in SYMS})

generated in 1s | hashes: {'BTCUSDT': 'ddf93c8cb7d69779', 'ETHUSDT': '614d4bf75b264457'}


symbol,BTCUSDT,ETHUSDT
split,,
test,36,36
train,80,80
val,30,30


scenario_family  abnormal_volume_burst  composite  fake_breakout_like  liquidity_stress_proxy  pump_and_dump_like  trade_flow_imbalance  wash_like_aggregate_proxy
symbol  split                                                                                                                                                     
BTCUSDT test                         5          4                   6                       5                   6                     5                          5
        train                       12          8                  12                      12                  12                    12                         12
        val                          5          3                   5                       4                   5                     4                          4
ETHUSDT test                         5          4                   6                       5                   6                     5                          5
        train                       12          8                  12                      12                  12                    12                         12
        val                          5          3                   5                       4                   5                     4                          4

events dropped by the placement budget: {'BTCUSDT': {'train': 0, 'val': 0, 'test': 0}, 'ETHUSDT': {'train': 0, 'val': 0, 'test': 0}}


## 3 · Write the model matrices, labels, state and event tables (staged)
Per symbol and split: `*_features.parquet` (datetime + the six features, **nothing else**), `*_labels.parquet` (labels and metadata, never model input), `*_state.parquet` (underlying aggregate state, used for recomputation checks).

In [7]:
STAGE.mkdir(parents=True, exist_ok=False)
for sub in ('btc', 'eth', 'events', 'metadata', 'validation', 'reports'): (STAGE / sub).mkdir()
FILES_SCHEMA = {}
for sym, tag in SYMS.items():
    for sp, (a, b) in BOUNDS.items():
        for kind in ('features', 'labels', 'state'):
            df = GEN[sym][kind].iloc[a:b].reset_index(drop=True); p = STAGE / tag / f'{sp}_{kind}.parquet'; df.to_parquet(p, index=False)
            FILES_SCHEMA[str(p.relative_to(STAGE))] = {'rows': len(df), 'columns': list(df.columns), 'dtypes': {c: str(t) for c, t in df.dtypes.items()}}
    GEN[sym]['events'].to_parquet(STAGE / 'events' / f'{tag}_events.parquet', index=False)
    FILES_SCHEMA[f'events/{tag}_events.parquet'] = {'rows': len(GEN[sym]['events']), 'columns': list(GEN[sym]['events'].columns), 'dtypes': {c: str(t) for c, t in GEN[sym]['events'].dtypes.items()}}
print('written:', len(FILES_SCHEMA), 'parquet files')

# summary tables required by the brief
LAB = {sym: GEN[sym]['labels'] for sym in SYMS}
SUMMARY = []
for sym in SYMS:
    for sp, (a, b) in BOUNDS.items():
        l = LAB[sym].iloc[a:b]; ev = GEN[sym]['events']; ev = ev[ev.split == sp]
        SUMMARY.append({'symbol': sym, 'split': sp, 'bars': b - a, 'events': len(ev), 'positive_bars': int(l.synthetic_label.sum()), 'positive_share': float(l.synthetic_label.mean()), 'dependency_tail_bars': int(l.dependency_tail.sum()),
                        'clean_normal_background_bars': int(l.normal_background.sum()), 'ood_events': int(ev.is_ood.sum()), 'composite_events': int(ev.composite.sum())})
SUMMARY = pd.DataFrame(SUMMARY); display(SUMMARY)
display(EV.groupby(['symbol', 'split', 'severity']).size().unstack(fill_value=0))
display(EV.groupby('split').duration_bars.describe())

written: 20 parquet files


,symbol,split,bars,events,positive_bars,positive_share,dependency_tail_bars,clean_normal_background_bars,ood_events,composite_events
0,BTCUSDT,train,90720,80,2682,0.0295635,1200,86838,0,8
1,BTCUSDT,val,18720,30,891,0.0475962,450,17379,0,3
2,BTCUSDT,test,20160,36,1672,0.0829365,540,17948,9,4
3,ETHUSDT,train,90720,80,2211,0.0243717,1200,87309,0,8
4,ETHUSDT,val,18720,30,845,0.0451389,450,17425,0,3
5,ETHUSDT,test,20160,36,1259,0.0624504,540,18361,9,4


severity       medium  severe  subtle
symbol  split                        
BTCUSDT test        7       7      22
        train      31      19      30
        val        12       6      12
ETHUSDT test       13       7      16
        train      34      21      25
        val        13       8       9

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,72,40.7083,32.576,7,20,28,51.25,144
train,160,30.5813,19.8423,6,15,25,41,84
val,60,28.9333,20.6684,5,12,20.5,40,88


## 4 · Scientific diagnostics (distribution realism · scenario diversity · detectability bias · leakage)
These are **not** model-performance results. The single-feature AUCs below only look for trivial synthetic signatures.

In [8]:
QS = [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
DIST, DIV, LEAK = {'method': 'event bars vs genuine Layer 1 clean background (TRAIN split, post warm-up, outside every dependency window)'}, {}, {}
def robust_z(x, ref):
    med, iqr = np.nanmedian(ref), np.subtract(*np.nanpercentile(ref, [75, 25])); return (x - med) / max(iqr / 1.349, 1e-12)
BG, EVB = {}, {}
for sym in SYMS:
    f = GEN[sym]['features']; l = GEN[sym]['labels']; a, b = BOUNDS['train']
    bgm = (l.normal_background.to_numpy() == 1) & (np.arange(len(l)) >= WARMUP) & (np.arange(len(l)) < b); BG[sym] = f.loc[bgm, FEATURES]
    EVB[sym] = {sp: f.loc[(l.synthetic_label.to_numpy() == 1) & (np.arange(len(l)) >= BOUNDS[sp][0]) & (np.arange(len(l)) < BOUNDS[sp][1]), FEATURES] for sp in SPL}
    DIST[sym] = {'background_quantiles': {ft: {str(q): float(np.quantile(BG[sym][ft], q)) for q in QS} for ft in FEATURES}, 'event_quantiles': {}, 'effect_sizes': {}, 'implausible': {}}
    for sp in SPL:
        ev_ = EVB[sym][sp]; DIST[sym]['event_quantiles'][sp] = {ft: {str(q): float(np.quantile(ev_[ft], q)) for q in QS} for ft in FEATURES}
        es = {}
        for ft in FEATURES:
            y = np.r_[np.zeros(min(len(BG[sym]), 200000)), np.ones(len(ev_))]; x = np.r_[BG[sym][ft].to_numpy()[:200000], ev_[ft].to_numpy()]; auc = auc_rank(x, y)
            es[ft] = {'auc_event_vs_background': auc, 'cliffs_delta': 2 * auc - 1, 'median_shift_robust_sd': float(np.median(robust_z(ev_[ft].to_numpy(), BG[sym][ft].to_numpy())))}
        DIST[sym]['effect_sizes'][sp] = es
    st = GEN[sym]['state']; ev_idx = np.flatnonzero(l.synthetic_label.to_numpy() == 1); fill = (st.volume / st.trades_count.clip(lower=1)).to_numpy(); bgfill = fill[bgm]
    lo_, hi_ = np.quantile(bgfill, [0.0005, 0.9995]); f_ev = fill[ev_idx]
    DIST[sym]['implausible'] = {'event_bars': int(len(ev_idx)), 'abs_log_return_gt_5pct': int((np.abs(f.loc[ev_idx, 'log_returns']) > 0.05).sum()), 'hl_ratio_gt_10pct': int((f.loc[ev_idx, 'hl_spread_ratio'] > 0.10).sum()),
                                'mean_fill_size_outside_0.5x_to_2x_real_extremes': int(((f_ev < lo_ * 0.5) | (f_ev > hi_ * 2)).sum()), 'real_max_abs_log_return': float(np.abs(ORIG[sym].log_returns).max()), 'real_max_hl': float(ORIG[sym].hl_spread_ratio.max()),
                                'event_max_abs_log_return': float(np.abs(f.loc[ev_idx, 'log_returns']).max()), 'event_max_hl': float(f.loc[ev_idx, 'hl_spread_ratio'].max()),
                                'event_max_volume_acceleration': float(f.loc[ev_idx, 'volume_acceleration'].max()), 'real_p999_volume_acceleration': float(np.quantile(BG[sym]['volume_acceleration'], .999))}
print(json.dumps({s: DIST[s]['implausible'] for s in SYMS}, indent=1, default=jsonable))
display(pd.DataFrame({sp: {ft: DIST['BTCUSDT']['effect_sizes'][sp][ft]['cliffs_delta'] for ft in FEATURES} for sp in SPL}).rename_axis('BTC Cliff delta (event vs background)'))

{
 "BTCUSDT": {
  "event_bars": 5245,
  "abs_log_return_gt_5pct": 0,
  "hl_ratio_gt_10pct": 0,
  "mean_fill_size_outside_0.5x_to_2x_real_extremes": 0,
  "real_max_abs_log_return": 0.015813143135901556,
  "real_max_hl": 0.01834082821133927,
  "event_max_abs_log_return": 0.010413367212928792,
  "event_max_hl": 0.016838902562424485,
  "event_max_volume_acceleration": 52.35494964640853,
  "real_p999_volume_acceleration": 21.899132754973884
 },
 "ETHUSDT": {
  "event_bars": 4315,
  "abs_log_return_gt_5pct": 0,
  "hl_ratio_gt_10pct": 0,
  "mean_fill_size_outside_0.5x_to_2x_real_extremes": 2,
  "real_max_abs_log_return": 0.038362282446787986,
  "real_max_hl": 0.08042700724925653,
  "event_max_abs_log_return": 0.009477497538505677,
  "event_max_hl": 0.01760763979124204,
  "event_max_volume_acceleration": 137.32435235530912,
  "real_p999_volume_acceleration": 26.94165999964798
 }
}


,train,val,test
BTC Cliff delta (event vs background),,,
log_returns,0.00214203,0.0117594,-0.00877944
trade_flow_imbalance,0.0119837,0.0494802,-0.0193591
hl_spread_ratio,0.258522,0.537047,0.441911
rush_order_density,-0.0479316,0.168048,0.174973
volume_acceleration,0.0387105,0.041989,0.0645951
large_trade_count,0.226086,0.265201,0.197738


In [9]:
# ---------- scenario diversity and detectability bias ----------
ALL_EV = EV.copy(); fam_rows = {}
DIV['counts'] = {'by_symbol_split_family': ALL_EV.groupby(['symbol', 'split', 'scenario_family']).size().rename('n').reset_index().to_dict(orient='records'), 'by_split_severity': ALL_EV.groupby(['split', 'severity']).size().rename('n').reset_index().to_dict(orient='records')}
DIV['duration'] = {sp: {'min': int(g.min()), 'p25': float(g.quantile(.25)), 'median': float(g.median()), 'p75': float(g.quantile(.75)), 'max': int(g.max()), 'n_unique': int(g.nunique())} for sp, g in ALL_EV.groupby('split').duration_bars}
DIV['shape_kinds'] = ALL_EV.groupby(['split', 'price_shape']).size().rename('n').reset_index().to_dict(orient='records')
DIV['amplitude_cv'] = {fam: {'peak_volume_multiplier_cv': float(g.peak_volume_multiplier.std() / g.peak_volume_multiplier.mean()), 'peak_abs_displacement_cv': float(g.peak_abs_log_displacement.std() / max(g.peak_abs_log_displacement.mean(), 1e-12)), 'duration_cv': float(g.duration_bars.std() / g.duration_bars.mean())} for fam, g in ALL_EV.groupby('scenario_family')}
DIV['context_coverage'] = {sp: {'sigma_pct_tertiles': np.histogram(g.sigma_pct.dropna(), bins=[0, 1 / 3, 2 / 3, 1.0001])[0].tolist(), 'volume_pct_tertiles': np.histogram(g.volume_pct.dropna(), bins=[0, 1 / 3, 2 / 3, 1.0001])[0].tolist()} for sp, g in ALL_EV.groupby('split')}
# family response profiles (median robust z of event bars by feature) and detectability (single-feature AUC vs clean background)
RESP, DET = {}, {}
for sym in SYMS:
    f = GEN[sym]['features']; l = GEN[sym]['labels']; n = len(l); bgm = (l.normal_background.to_numpy() == 1) & (np.arange(n) >= WARMUP) & (np.arange(n) < BOUNDS['train'][1])
    for col in LABEL_COLS:
        m = (l[col].to_numpy() == 1) & (np.arange(n) < BOUNDS['val'][1])      # train + validation bars only for diagnostics
        if m.sum() < 30: continue
        z = {ft: float(np.median(robust_z(f.loc[m, ft].to_numpy(), BG[sym][ft].to_numpy()))) for ft in FEATURES}; RESP[(sym, col)] = z
        aucs = {}
        for ft in FEATURES:
            x = np.r_[BG[sym][ft].to_numpy()[:200000], f.loc[m, ft].to_numpy()]; y = np.r_[np.zeros(min(len(BG[sym]), 200000)), np.ones(int(m.sum()))]; a_ = auc_rank(x, y); aucs[ft] = max(a_, 1 - a_)
        zmax = np.max(np.abs(np.column_stack([robust_z(f.loc[m, ft].to_numpy(), BG[sym][ft].to_numpy()) for ft in FEATURES])), axis=1); zb = np.max(np.abs(np.column_stack([robust_z(BG[sym][ft].to_numpy()[:200000], BG[sym][ft].to_numpy()) for ft in FEATURES])), axis=1)
        a2 = auc_rank(np.r_[zb, zmax], np.r_[np.zeros(len(zb)), np.ones(len(zmax))]); DET[(sym, col)] = {'bars': int(m.sum()), 'single_feature_auc': aucs, 'max_single_feature_auc': float(max(aucs.values())), 'max_abs_z_rule_auc': a2}
DIV['family_response_median_robust_z'] = {f'{k[0]}|{k[1]}': v for k, v in RESP.items()}
DIV['detectability_bias'] = {f'{k[0]}|{k[1]}': v for k, v in DET.items()}
vecs = {k: np.array([v[ft] for ft in FEATURES]) for k, v in RESP.items() if k[0] == 'BTCUSDT'}
names = list(vecs); cs = np.array([[float(np.dot(vecs[a], vecs[b]) / max(np.linalg.norm(vecs[a]) * np.linalg.norm(vecs[b]), 1e-12)) for b in names] for a in names])
DIV['family_response_cosine_similarity_BTC'] = pd.DataFrame(cs, index=[n[1] for n in names], columns=[n[1] for n in names]).round(3).to_dict()
display(pd.DataFrame({f'{k[0][:3]}|{k[1][6:]}': v['single_feature_auc'] for k, v in DET.items()}).T.round(3))
TRIVIAL = {f'{k[0]}|{k[1]}': v['max_single_feature_auc'] for k, v in DET.items() if v['max_single_feature_auc'] > CFG['trivial_signature_single_feature_auc']}
print('families with a trivial single-feature signature (AUC >', CFG['trivial_signature_single_feature_auc'], '):', TRIVIAL or 'none')

# ---------- leakage / isolation between splits ----------
def fingerprint(g): return np.column_stack([np.log(g.duration_bars), np.log1p(g.peak_abs_log_displacement / g.sigma_local), np.log(g.peak_volume_multiplier), g.peak_flow_shift, np.log(g.peak_range_multiplier), np.log(g.peak_trade_multiplier), g.composite])
FP = {}
for sym in SYMS:
    g = ALL_EV[ALL_EV.symbol == sym]; tr_ = g[g.split == 'train']; mu, sd = fingerprint(tr_).mean(axis=0), fingerprint(tr_).std(axis=0) + 1e-9
    for sp in ('val', 'test'):
        q = (fingerprint(g[g.split == sp]) - mu) / sd; t = (fingerprint(tr_) - mu) / sd; dmin = np.sqrt(((q[:, None, :] - t[None, :, :]) ** 2).sum(-1)).min(axis=1)
        FP[f'{sym}|{sp}'] = {'min_distance_to_nearest_train_event': float(dmin.min()), 'median_distance': float(np.median(dmin)), 'median_distance_ood': float(np.median(dmin[g[g.split == sp].is_ood.to_numpy() == 1])) if (g[g.split == sp].is_ood == 1).any() else None,
                             'median_distance_in_distribution': float(np.median(dmin[g[g.split == sp].is_ood.to_numpy() == 0]))}
LEAK['fingerprint_nearest_train_event'] = FP
LEAK['seeds'] = {'n_events': int(len(ALL_EV)), 'n_unique_seeds': int(ALL_EV.random_seed.nunique()), 'n_unique_param_hashes': int(ALL_EV.params_hash.nunique()), 'n_unique_event_ids': int(ALL_EV.event_id.nunique())}
LEAK['seed_overlap_across_splits'] = int(len(set(ALL_EV[ALL_EV.split == 'train'].random_seed) & (set(ALL_EV[ALL_EV.split == 'val'].random_seed) | set(ALL_EV[ALL_EV.split == 'test'].random_seed))) + len(set(ALL_EV[ALL_EV.split == 'val'].random_seed) & set(ALL_EV[ALL_EV.split == 'test'].random_seed)))
print(json.dumps(LEAK, indent=1, default=jsonable)[:1800])

,log_returns,trade_flow_imbalance,hl_spread_ratio,rush_order_density,volume_acceleration,large_trade_count
BTC|pump_and_dump_like,0.501,0.501,0.708,0.537,0.514,0.676
BTC|fake_breakout_like,0.508,0.541,0.721,0.51,0.541,0.635
BTC|abnormal_volume_burst,0.505,0.525,0.742,0.583,0.564,0.797
BTC|trade_flow_imbalance,0.502,0.521,0.616,0.507,0.508,0.579
BTC|wash_like_aggregate_proxy,0.506,0.514,0.557,0.635,0.502,0.533
BTC|liquidity_stress_proxy,0.502,0.509,0.781,0.57,0.52,0.652
ETH|pump_and_dump_like,0.501,0.505,0.736,0.563,0.539,0.616
ETH|fake_breakout_like,0.526,0.501,0.789,0.502,0.527,0.679
ETH|abnormal_volume_burst,0.5,0.554,0.641,0.568,0.567,0.645
ETH|trade_flow_imbalance,0.502,0.538,0.615,0.52,0.518,0.555


families with a trivial single-feature signature (AUC > 0.95 ): none
{
 "fingerprint_nearest_train_event": {
  "BTCUSDT|val": {
   "min_distance_to_nearest_train_event": 0.5037769594380511,
   "median_distance": 0.9272391554104765,
   "median_distance_ood": null,
   "median_distance_in_distribution": 0.9272391554104765
  },
  "BTCUSDT|test": {
   "min_distance_to_nearest_train_event": 0.4341367122582031,
   "median_distance": 0.856036216987216,
   "median_distance_ood": 1.1440040127570483,
   "median_distance_in_distribution": 0.835294066726195
  },
  "ETHUSDT|val": {
   "min_distance_to_nearest_train_event": 0.3911827866233519,
   "median_distance": 0.9361035769471311,
   "median_distance_ood": null,
   "median_distance_in_distribution": 0.9361035769471311
  },
  "ETHUSDT|test": {
   "min_distance_to_nearest_train_event": 0.40749957932293546,
   "median_distance": 0.9682593552599967,
   "median_distance_ood": 1.3094309929167924,
   "median_distance_in_distribution": 0.9449471209974435

## 5 · Metadata, reports and file manifest

In [10]:
dump_json({**CFG, 'symbols': list(SYMS), 'splits': {k: list(v) for k, v in BOUNDS.items()}, 'warmup_bars': WARMUP, 'derived_buy_sell': 'buy = V*(1+tfi)/2 ; sell = V - buy (NOT observed)', 'large_trade_threshold': THR, 'volume_acceleration_floor': FLOOR,
           'seeding': 'SeedSequence([master_seed, symbol_index, split_index]) -> per-split rng (specs, placement); per-event rng = default_rng(event_seed)', 'run_started_utc': RUN_STARTED}, STAGE / 'metadata' / 'generation_config.json')
dump_json({'feature_order': FEATURES, 'contract': 'tradesense_c3_layer1_feature_contract v1.0.0 (embedded copy, cross-checked)', 'contract_file_on_drive': CONTRACT_FILE, 'constants_used': {'large_trade_threshold': THR, 'volume_acceleration_floor': FLOOR},
           'formulas': {'log_returns': 'ln(close_t/close_{t-1}); bar 0 = 0', 'trade_flow_imbalance': '(buy-sell)/(buy+sell+1e-8), 0 if buy+sell == 0', 'hl_spread_ratio': '(high-low)/close  [NOT a bid-ask spread]', 'rush_order_density': 'trades_count/(volume+1e-8)  [NOT order submission]',
                        'volume_acceleration': '(volume_t - mean15)/max(std15, floor); previous 15 closed bars (shift 1), ddof=1; bars 0..14 = 0.0 (masked)', 'large_trade_count': 'executed fills with quantity >= threshold'},
           'layer1_formula_verification': L1_CHECK, 'discrepancies_and_notes': DISCREPANCIES}, STAGE / 'metadata' / 'feature_contract_reference.json')
SCEN = {'A': 'Temporary price rise with volume/taker-buy response followed by a variable reversal (optional undershoot). NOT verified pump-and-dump intent.',
        'B': 'Excursion beyond the pre-event local high/low (reference window W<=120 bars before the event, original data only) that closes beyond it at least once, then returns inside the earlier range.',
        'C': 'Unusual volume and executed-trade burst with small, random-sign price response; subtle cases overlap normal volume variation (not every spike is manipulation).',
        'D': 'Persistent one-sided taker pressure with partial price impact that relaxes by the end of the event.',
        'E': 'Wash-like AGGREGATE proxy: elevated fill count and volume with taker share pulled toward balance and little net price movement. Not verified wash trading (no participant identities exist).',
        'F': 'Stress-like price/trade pattern: noisy displacement, strong candle-range expansion, moderate volume, erratic flow. Not an order-book liquidity-withdrawal claim.',
        'G': 'Composite of a price-driving primary family with a non-price secondary family; only pairs in COMPAT; every contributing label recorded.'}
dump_json({'families': {FAMS.get(k, 'composite'): v for k, v in SCEN.items()}, 'compatible_composites': COMPAT, 'phases': PHASES, 'severity_ranges': CFG['severity_ranges'], 'ood_definition': 'test-only: late peaks (0.75-0.92), steep rises (a 2.5-4.5), slow recovery (b 0.3-0.6), amplitudes x1.4, durations x1.5-2.2 (cap 160)',
           'not_implemented_by_design': ['spoofing', 'layering', 'order-book liquidity withdrawal (no order-book evidence in the six-feature contract; TGNN track)']}, STAGE / 'metadata' / 'scenario_definitions.json')
dump_json({'bounds_bar_index': {k: list(v) for k, v in BOUNDS.items()}, 'split_days': {'train': '2024-05-01..2024-07-02', 'val': '2024-07-03..2024-07-15', 'test': '2024-07-16..2024-07-29'}, 'rows_per_symbol_split': {sp: b - a for sp, (a, b) in BOUNDS.items()},
           'event_margins': {'pad_start_bars': PAD_S, 'pad_end_bars': PAD_E, 'min_gap_bars': MIN_GAP, 'dependency_tail_bars': TAIL}, 'events_dropped_by_placement_budget': {s: GEN[s]['dropped'] for s in SYMS}, 'summary': SUMMARY.to_dict(orient='records'),
           'normal_background_reference': {'definition': 'labels.normal_background == 1: genuine Layer 1 bars outside every event window and outside the 15-bar dependency tail; values identical to ' + SRC_VERSION, 'rows': {f'{s}|{sp}': int(LAB[s].iloc[a:b].normal_background.sum()) for s in SYMS for sp, (a, b) in BOUNDS.items()}},
           'use_policy': {'autoencoder': 'train on GENUINE Layer 1 only; never on Layer 3', 'xgboost_transformer': 'synthetic experiments; fit on train, select on val, report test once', 'tgnn': 'separate order-book track'}}, STAGE / 'metadata' / 'split_manifest.json')
dump_json(DIST, STAGE / 'validation' / 'distribution_report.json'); dump_json(DIV, STAGE / 'validation' / 'diversity_report.json'); dump_json(LEAK, STAGE / 'validation' / 'leakage_report.json')

LIMITS = ['Labels describe the synthetic INJECTION MECHANISM; they are not verified real-world manipulation and results on them do not prove real-world detection.',
          'Synthesis is at the AGGREGATE (1-minute) level. Tick-level physical consistency is NOT claimed. Buy/sell volume is derived from trade_flow_imbalance; large-fill counts are scaled and capped by testable bounds (<= trades, x threshold <= volume) but fill-size distributions are not simulated.',
          'trade_flow_imbalance and large_trade_count remain PENDING raw-tick reconciliation in the Layer 1 contract; Layer 3 inherits that status.',
          'Wash-like and liquidity-stress scenarios are aggregate proxies; genuine wash trading needs participant identities and liquidity withdrawal needs order-book data. Spoofing/layering are not represented.',
          'Per-split event counts are modest (val %d, test %d per symbol); event-level metrics must carry confidence intervals. Bars within one event are not independent.' % (CFG['events_per_symbol_split']['val'], CFG['events_per_symbol_split']['test']),
          'BTC and ETH events are generated independently (no cross-asset co-movement is modelled).',
          'Out-of-distribution test events probe robustness to shape/scale shifts; their ranges are fixed in generation_config.json before training and are one specific, arbitrary notion of "unseen".',
          'The 15 bars after each event carry altered volume_acceleration (dependency tail) and are excluded from the clean background via normal_background = 0.']
(STAGE / 'reports' / 'limitations.md').write_text('# Limitations\n\n' + '\n'.join('- ' + l for l in LIMITS) + '\n\n## Discrepancies / notes vs the task text\n\n' + '\n'.join('- ' + d for d in DISCREPANCIES) + '\n', encoding='utf-8')
(STAGE / 'reports' / 'research_methodology_summary.md').write_text('\n'.join(['# Layer 3 v2 methodology', '',
    'Source: genuine Layer 1 BTCUSDT/ETHUSDT 1-minute matrices (%s). For each event a coherent aggregate state (price displacement, volume, taker share, fill count, large fills, candle range) is drawn from bounded stochastic distributions and applied only inside the event window; the six features are then recomputed with the exact Layer 1 formulas.' % SRC_VERSION, '',
    'Isolation: per-split generation, 250-bar start margin, 60-bar end margin, >=150 bars between events, 15-bar dependency tail, independent seeds; test includes an out-of-distribution share.', '',
    'Families: ' + '; '.join(f'{k} {v}' for k, v in FAMS.items()) + '; composites of compatible pairs.', '', 'Validation: 20 automated tests plus distribution-realism, diversity, detectability-bias and leakage diagnostics; see validation/.', '',
    'Training policy: Autoencoder on genuine Layer 1 only; Layer 3 is for evaluation and supervised synthetic experiments; labels/metadata are never model inputs.']), encoding='utf-8')
(STAGE / 'reports' / 'README.md').write_text('\n'.join(['# processed_layer3_final_v2', '', 'Synthetic behavioural-scenario evaluation dataset on genuine Layer 1 data. ' + GEN_VERSION, '',
    '- `{btc,eth}/{train,val,test}_features.parquet` : datetime + the six features (model input, exact contract order)', '- `{btc,eth}/{split}_labels.parquet` : labels/metadata (NEVER model input)', '- `{btc,eth}/{split}_state.parquet` : aggregate state used for recomputation checks',
    '- `events/{btc,eth}_events.parquet` : event-level metadata', '- `metadata/` : config, contract reference, scenario definitions, split manifest, file manifest', '- `validation/` : validation, distribution, diversity and leakage reports', '- `reports/` : this README, methodology, limitations', '',
    'Frozen sources (Layer 1, Layer 2 v3, Layer 2 model-ready v1, Layer 3 v1) are not modified. Read reports/limitations.md before using the data.']), encoding='utf-8')

def build_manifest():
    files = {}
    for p in sorted(STAGE.rglob('*')):
        if p.is_file() and p.name not in ('manifest.json', 'validation_report.json'):
            rel = str(p.relative_to(STAGE)); e = {'size': p.stat().st_size, 'sha256': sha256_file(p)}
            if p.suffix == '.parquet': m = pq.read_metadata(p); e.update({'rows': m.num_rows, 'columns': [m.schema.names[i] for i in range(len(m.schema.names))]})
            files[rel] = e
    return files
MANIFEST = {'dataset': 'processed_layer3_final_v2', 'generation_version': GEN_VERSION, 'source_dataset_version': SRC_VERSION, 'excluded_from_manifest': ['manifest.json', 'validation_report.json'], 'files': build_manifest()}
dump_json(MANIFEST, STAGE / 'metadata' / 'manifest.json'); print('manifest entries:', len(MANIFEST['files']))

manifest entries: 30


## 6 · The 20 automated validation tests (new artifacts only) + scientific gates
All tests run and are reported; nothing is weakened. Any failure makes the verdict **BLOCKED** and nothing is published.

In [11]:
RES = []
def T(n, name, ok, detail=''):
    RES.append({'test': n, 'name': name, 'pass': bool(ok), 'detail': str(detail)[:300]}); print(('PASS ' if ok else 'FAIL ') + f'T{n:02d} {name}' + ('' if ok else f'  <-- {str(detail)[:200]}'))
def rd(sym, sp, kind): return pd.read_parquet(STAGE / SYMS[sym] / f'{sp}_{kind}.parquet')
FE = {(s, sp): rd(s, sp, 'features') for s in SYMS for sp in SPL}; LB = {(s, sp): rd(s, sp, 'labels') for s in SYMS for sp in SPL}; ST = {(s, sp): rd(s, sp, 'state') for s in SYMS for sp in SPL}
def full(D, s): return pd.concat([D[(s, sp)] for sp in SPL], ignore_index=True)

req = [f'{SYMS[s]}/{sp}_{k}.parquet' for s in SYMS for sp in SPL for k in ('features', 'labels', 'state')] + [f'events/{SYMS[s]}_events.parquet' for s in SYMS] + \
      ['metadata/generation_config.json', 'metadata/feature_contract_reference.json', 'metadata/scenario_definitions.json', 'metadata/split_manifest.json', 'metadata/manifest.json', 'validation/distribution_report.json', 'validation/diversity_report.json', 'validation/leakage_report.json', 'reports/README.md', 'reports/research_methodology_summary.md', 'reports/limitations.md']
T(1, 'all required files exist', all((STAGE / r).is_file() for r in req), [r for r in req if not (STAGE / r).is_file()])
T(2, 'frozen source datasets unchanged (sha256, size, mtime)', {n: pin_dir(BASE / n) for n in PIN} == PIN)
T(3, 'feature names and order (features files = datetime + contract order; dtypes as Layer 1)', all(list(FE[k].columns) == ['datetime'] + FEATURES for k in FE) and all(all(str(FE[(s, sp)][f].dtype) == FEATURE_DTYPES[s][f] for f in FEATURES) for s in SYMS for sp in SPL))
tsb = {s: full(FE, s).datetime for s in SYMS}
T(4, 'BTC/ETH timestamps identical, 1-min grid, equal to Layer 1', (tsb['BTCUSDT'] == tsb['ETHUSDT']).all() and all((tsb[s].reset_index(drop=True) == ORIG[s].datetime).all() for s in SYMS) and all((tsb[s].diff().dropna() == pd.Timedelta(minutes=1)).all() for s in SYMS))
ok5, d5 = True, []
for s in SYMS:
    for sp, (a, b) in BOUNDS.items():
        d = FE[(s, sp)].datetime; ok = len(d) == b - a and (d.reset_index(drop=True) == ORIG[s].datetime.iloc[a:b].reset_index(drop=True)).all(); ok5 &= ok; d5 += ([] if ok else [f'{s}/{sp}'])
    ok5 &= FE[(s, 'train')].datetime.max() < FE[(s, 'val')].datetime.min() and FE[(s, 'val')].datetime.max() < FE[(s, 'test')].datetime.min()
T(5, 'chronological split membership (counts and timestamps equal Layer 1 partitions)', ok5, d5)
ok6, d6 = True, []
for s in SYMS:
    ev = EV[EV.symbol == s].sort_values('start_idx').reset_index(drop=True)
    for _, r in ev.iterrows():
        a, b = BOUNDS[r.split]; ok = (r.start_idx >= a + PAD_S) and (r.dependency_end_idx <= b - 1 - PAD_E); ok6 &= ok; d6 += ([] if ok else [r.event_id])
    gaps = ev.start_idx.to_numpy()[1:] - ev.dependency_end_idx.to_numpy()[:-1]; ok6 &= bool((gaps > MIN_GAP).all()) if len(gaps) else True
T(6, 'no event or dependency tail crosses/touches a split boundary or another event (margins and spacing hold)', ok6, d6)
T(7, 'event IDs unique', EV.event_id.is_unique and all(r.event_id.startswith(r.symbol + '-' + r.split + '-') for r in EV.itertuples()))
T(8, 'no duplicate scenario parameter sets or seeds (also across splits)', EV.params_hash.is_unique and EV.random_seed.is_unique and LEAK['seed_overlap_across_splits'] == 0, LEAK['seeds'])
fin = all(np.isfinite(FE[k][FEATURES].to_numpy(float)).all() for k in FE)
T(9, 'all model features finite', fin)
dom = True
for k in FE:
    f = FE[k]; dom &= bool(f.trade_flow_imbalance.between(-1, 1).all() and (f.hl_spread_ratio >= 0).all() and (f.hl_spread_ratio < 1).all() and (f.rush_order_density >= 0).all() and (f.large_trade_count >= 0).all() and np.allclose(f.large_trade_count, np.round(f.large_trade_count)) and (f.log_returns.abs() < 0.2).all())
T(10, 'feature domain constraints (tfi in [-1,1], hl in [0,1), rush>=0, large nonneg integer, |log return|<0.2)', dom)
ok11, d11 = True, []
for s in SYMS:
    st = full(ST, s); lb = full(LB, s); ev = lb.synthetic_label.to_numpy() == 1; thr = THR[s]
    c = {'prices>0': (st[['open', 'high', 'low', 'close']] > 0).all().all(), 'high>=max(o,c)': (st.high >= st[['open', 'close']].max(axis=1) - 1e-12).all(), 'low<=min(o,c)': (st.low <= st[['open', 'close']].min(axis=1) + 1e-12).all(), 'high>=low': (st.high >= st.low).all(),
         'volume>=0': (st.volume >= 0).all(), 'trades int>=0': bool((st.trades_count >= 0).all()), 'large int>=0': bool((st.large_trade_count >= 0).all()), 'large<=trades': (st.large_trade_count <= st.trades_count).all(),
         'buy,sell>=0': bool((st.buy_volume_derived >= -1e-12).all() and (st.sell_volume_derived >= -1e-12).all()), 'buy+sell==volume': np.allclose(st.buy_volume_derived + st.sell_volume_derived, st.volume, rtol=1e-9, atol=1e-12),
         'event bars: large*thr<=volume': bool((st.large_trade_count[ev] * thr <= st.volume[ev] * (1 + 1e-9)).all())}
    for k_, v in c.items():
        if not v: ok11 = False; d11.append(f'{s}:{k_}')
T(11, 'synthetic OHLC / volume / trade / buy-sell state consistency', ok11, d11)
ok12, d12 = True, []
for s in SYMS:
    st = full(ST, s); fe = full(FE, s); lb = full(LB, s); c = st.close.to_numpy(float); lr = np.r_[0.0, np.log(c[1:] / c[:-1])]
    hl = (st.high - st.low) / st.close; rush = st.trades_count / (st.volume + 1e-8); va = vol_acc(st.volume.to_numpy(float), FLOOR[s]); evm = lb.synthetic_label.to_numpy() == 1
    chk_ = {'log_returns': np.allclose(lr, fe.log_returns, rtol=0, atol=1e-12), 'hl': np.allclose(hl, fe.hl_spread_ratio, rtol=1e-9, atol=1e-13), 'rush': np.allclose(rush, fe.rush_order_density, rtol=1e-9, atol=0),
            'volume_acceleration(all bars>=15)': np.allclose(va[WARMUP:], fe.volume_acceleration.to_numpy()[WARMUP:], rtol=1e-7, atol=1e-8), 'tfi(event bars, exact formula)': np.allclose(derived_tfi(st.buy_volume_derived.to_numpy(), st.sell_volume_derived.to_numpy())[evm], fe.trade_flow_imbalance.to_numpy()[evm], rtol=1e-12, atol=1e-13),
            'large==state': np.array_equal(fe.large_trade_count.to_numpy(float), st.large_trade_count.to_numpy(float))}
    for k_, v in chk_.items():
        if not v: ok12 = False; d12.append(f'{s}:{k_}')
T(12, 'exact rolling/feature recomputation from the saved state (Layer 1 formulas, verified floors)', ok12, d12)
ok13, d13 = True, []
for s in SYMS:
    lb = full(LB, s); ev = EV[EV.symbol == s]; m = np.zeros(len(lb), bool); eidc = lb.event_id.to_numpy()
    for r in ev.itertuples():
        seg = slice(r.start_idx, r.end_idx + 1); m[seg] = True
        if not ((lb.synthetic_label.to_numpy()[seg] == 1).all() and (eidc[seg] == r.event_id).all() and lb.datetime.iloc[r.start_idx] == r.start_timestamp and lb.datetime.iloc[r.end_idx] == r.end_timestamp and r.end_idx - r.start_idx + 1 == r.duration_bars): ok13 = False; d13.append(r.event_id)
        fams = json.loads(r.families_json)
        if not all(lb[LABEL_COLS[ord(f) - 65]].to_numpy()[seg].all() for f in fams) or lb[LABEL_COLS].to_numpy()[seg].sum(axis=1).max() != len(fams): ok13 = False; d13.append(r.event_id + ':multilabel')
    if not (np.array_equal(m, lb.synthetic_label.to_numpy() == 1) and (lb.normal_background.to_numpy()[m] == 0).all() and (lb.synthetic_label.to_numpy()[lb.dependency_tail.to_numpy() == 1] == 0).all()): ok13 = False; d13.append(s + ':outside')
T(13, 'labels align with event start/end, multi-label columns, tails and background flags', ok13, d13[:5])
ok14, d14 = True, []
for s in SYMS:
    o = ORIG[s]; fe = full(FE, s); st = full(ST, s); lb = full(LB, s); bg = lb.normal_background.to_numpy() == 1; tail = lb.dependency_tail.to_numpy() == 1; idx = np.arange(len(o))
    for ft in FEATURES:
        m = bg & ((idx >= WARMUP) if ft == 'volume_acceleration' else True)
        if not np.array_equal(fe[ft].to_numpy()[m], o[ft].to_numpy()[m]): ok14 = False; d14.append(f'{s}:{ft}:background')
        mt = tail if ft == 'volume_acceleration' else np.zeros(len(o), bool)
        if ft != 'volume_acceleration' and not np.array_equal(fe[ft].to_numpy()[tail], o[ft].to_numpy()[tail]): ok14 = False; d14.append(f'{s}:{ft}:tail')
    for c_ in ('open', 'high', 'low', 'close', 'volume', 'trades_count'):
        if not np.array_equal(st[c_].to_numpy()[bg | tail], o[c_].to_numpy()[bg | tail]): ok14 = False; d14.append(f'{s}:{c_}:state')
T(14, 'untouched background (and non-volume features in tails) are bit-identical to Layer 1', ok14, d14)
dv = ALL_EV.groupby('split').agg(n=('event_id', 'size'), fams=('scenario_family', 'nunique'), sev=('severity', 'nunique'), dur=('duration_bars', 'nunique'), shapes=('price_shape', 'nunique')); sigt = ALL_EV.groupby('split').sigma_pct.apply(lambda g: np.histogram(g.dropna(), bins=[0, 1 / 3, 2 / 3, 1.0001])[0].min() / max(len(g), 1))
amp_cv = ALL_EV.groupby('scenario_family').peak_volume_multiplier.apply(lambda g: g.std() / g.mean())
ok15 = bool((dv.fams >= 6).all() and (dv.sev == 3).all() and (dv.dur >= 8).all() and (dv.shapes >= 3).all() and (amp_cv > 0.2).all() and (sigt > 0.1).all())
T(15, 'scenario diversity (all families+composites, 3 severities, many durations/shapes, amplitude CV>0.2, volatility-regime coverage)', ok15, f'{dv.to_dict()} cv_min={amp_cv.min():.2f} min_tertile_share={sigt.min():.2f}')
GEN2 = {sym: generate_symbol(i, sym) for i, sym in enumerate(SYMS)}; H2 = {sym: gen_hash(GEN2[sym]) for sym in SYMS}
T(16, 'reproducible from config + seeds (full regeneration gives identical hashes)', H2 == GEN_HASH, {k: v[:12] for k, v in H2.items()}); del GEN2
cols_ok = all(list(FE[k].columns) == ['datetime'] + FEATURES for k in FE) and all(not (set(FEATURES) & set(LB[k].columns)) for k in LB) and all(('event_id' not in FE[k].columns) for k in FE)
T(17, 'model features contain no labels/event IDs/metadata; labels live in separate files', cols_ok)
man = json.loads((STAGE / 'metadata' / 'manifest.json').read_text())['files']; actual = {str(p.relative_to(STAGE)) for p in STAGE.rglob('*') if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
T(18, 'manifest describes every file (sha256, size, rows, columns)', set(man) == actual and all(sha256_file(STAGE / k) == v['sha256'] and (STAGE / k).stat().st_size == v['size'] for k, v in man.items()), sorted(actual ^ set(man))[:5])
fpmin = min(v['min_distance_to_nearest_train_event'] for v in FP.values())
T(19, 'validation/test scenario realizations isolated from training (no shared seeds/params, nearest-train-event distance > 0)', LEAK['seed_overlap_across_splits'] == 0 and EV.params_hash.is_unique and fpmin > CFG['fingerprint_min_distance'], f'min fingerprint distance {fpmin:.3f}')
ok20 = True
for rel, sc in FILES_SCHEMA.items():
    m = pq.read_metadata(STAGE / rel); ok20 &= (m.num_rows == sc['rows'] and list(m.schema.names) == sc['columns'] and man[rel]['rows'] == sc['rows'])
T(20, 'files readable and consistent with reported schemas', ok20)

# ---------------- pre-registered scientific gates (separate from the 20 tests) ----------------
SCI = []
SCI.append({'gate': 'no trivial single-feature signature (max single-feature AUC <= %.2f per family)' % CFG['trivial_signature_single_feature_auc'], 'pass': not TRIVIAL, 'detail': TRIVIAL or 'none'})
imp = {s: DIST[s]['implausible'] for s in SYMS}
SCI.append({'gate': 'event extremes within physical plausibility (|log return| <= 2x real max, hl <= 2x real max)', 'pass': all(imp[s]['event_max_abs_log_return'] <= 2 * imp[s]['real_max_abs_log_return'] and imp[s]['event_max_hl'] <= 2 * imp[s]['real_max_hl'] for s in SYMS), 'detail': {s: [imp[s]['event_max_abs_log_return'], imp[s]['real_max_abs_log_return'], imp[s]['event_max_hl'], imp[s]['real_max_hl']] for s in SYMS}})
fb = EV[EV.scenario_type.str.contains('fake_breakout')]; fbp = [json.loads(p) for p in fb.params_json]
ret_in = np.mean([c['params'].get('returns_inside_range', False) for p in fbp for c in p['components'] if c['family'] == 'B']); brk = np.mean([c['params'].get('bars_closing_beyond_ref', 0) >= 1 for p in fbp for c in p['components'] if c['family'] == 'B'])
SCI.append({'gate': 'fake-breakout events: close beyond the pre-event reference level and return inside (>=90%)', 'pass': bool(ret_in >= 0.9 and brk >= 0.99), 'detail': {'returned_inside_share': float(ret_in), 'closed_beyond_share': float(brk), 'n': len(fb)}})
for g_ in SCI: print(('PASS ' if g_['pass'] else 'FAIL ') + 'SCI ' + g_['gate'], '' if g_['pass'] else g_['detail'])
ALL_PASS = all(r['pass'] for r in RES) and all(g_['pass'] for g_ in SCI)
VERDICT = 'READY WITH DOCUMENTED LIMITATIONS' if ALL_PASS else 'BLOCKED'
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} tests passed | scientific gates {sum(g_['pass'] for g_ in SCI)}/{len(SCI)} | VERDICT: {VERDICT}")

PASS T01 all required files exist
PASS T02 frozen source datasets unchanged (sha256, size, mtime)
PASS T03 feature names and order (features files = datetime + contract order; dtypes as Layer 1)
PASS T04 BTC/ETH timestamps identical, 1-min grid, equal to Layer 1
PASS T05 chronological split membership (counts and timestamps equal Layer 1 partitions)
PASS T06 no event or dependency tail crosses/touches a split boundary or another event (margins and spacing hold)
PASS T07 event IDs unique
PASS T08 no duplicate scenario parameter sets or seeds (also across splits)
PASS T09 all model features finite
PASS T10 feature domain constraints (tfi in [-1,1], hl in [0,1), rush>=0, large nonneg integer, |log return|<0.2)
PASS T11 synthetic OHLC / volume / trade / buy-sell state consistency
PASS T12 exact rolling/feature recomputation from the saved state (Layer 1 formulas, verified floors)
PASS T13 labels align with event start/end, multi-label columns, tails and background flags
PASS T14 untouched 

## 7 · Publish (atomic) and SHARE BLOCK

In [12]:
dump_json({'run_started_utc': RUN_STARTED, 'generation_version': GEN_VERSION, 'tests': RES, 'scientific_gates': SCI, 'verdict': VERDICT, 'note': 'Executed on real Drive data only if BASE is the Drive root; mock runs are development tests.'}, STAGE / 'validation' / 'validation_report.json')
assert {n: pin_dir(BASE / n) for n in PIN} == PIN, 'frozen inputs changed during the run'
if ALL_PASS:
    assert not OUT_FINAL.exists(); os.rename(STAGE, OUT_FINAL); where = OUT_FINAL; print('PUBLISHED ->', where)
else:
    where = STAGE; print('NOT PUBLISHED (a test failed). Staging folder kept for inspection:', where)

fcount = sorted(str(p.relative_to(where)) for p in where.rglob('*') if p.is_file())
ev_tab = EV.groupby(['symbol', 'split', 'scenario_family']).size().unstack(fill_value=0)
print('\n==================== LAYER 3 V2 — FINAL SHARE BLOCK ====================')
print('1. Output directory:', where)
print('2. Files created:', len(fcount), '(btc/eth x train/val/test x features/labels/state, 2 event tables, metadata, validation, reports)')
print('3. Rows per symbol:', {sym: {sp: b - a for sp, (a, b) in BOUNDS.items()} for sym in SYMS})
print('4. Scenario families:', list(FAMS.values()) + ['composite'], '| compatible composite pairs:', COMPAT)
print('5. Event counts (symbol, split, family):'); print(ev_tab.to_string())
print('6. Positive / clean-normal / tail bars:'); print(SUMMARY[['symbol', 'split', 'positive_bars', 'positive_share', 'clean_normal_background_bars', 'dependency_tail_bars', 'ood_events']].to_string(index=False))
print('7. Feature consistency (T09-T12):', {r['test']: r['pass'] for r in RES if r['test'] in (9, 10, 11, 12)})
print('8. Temporal validation (T04-T05):', {r['test']: r['pass'] for r in RES if r['test'] in (4, 5)})
print('9. Event isolation (T06-T08, T19):', {r['test']: r['pass'] for r in RES if r['test'] in (6, 7, 8, 19)}, '| min fingerprint distance val/test->train:', round(fpmin, 3))
print('10. Reproducibility (T16):', [r['pass'] for r in RES if r['test'] == 16][0], {k: v[:12] for k, v in GEN_HASH.items()})
print('11. Distribution realism: max |Cliff delta| per feature (event vs clean real background, TRAIN):', {ft: round(max(abs(DIST[s]['effect_sizes']['train'][ft]['cliffs_delta']) for s in SYMS), 2) for ft in FEATURES}, '| implausible:', {s: {k: DIST[s]['implausible'][k] for k in ('abs_log_return_gt_5pct', 'hl_ratio_gt_10pct', 'mean_fill_size_outside_0.5x_to_2x_real_extremes')} for s in SYMS})
print('12. Leakage validation:', {'seed_overlap': LEAK['seed_overlap_across_splits'], 'unique_param_hashes': LEAK['seeds']['n_unique_param_hashes'] == LEAK['seeds']['n_events'], 'trivial_signature_families': list(TRIVIAL) or 'none'})
print('13. Frozen source integrity (T02):', [r['pass'] for r in RES if r['test'] == 2][0], '| pinned dirs:', list(PIN))
print('14. Remaining limitations: aggregate-level synthesis (no tick-level consistency); derived buy/sell; TFI/large-fill raw-tick reconciliation pending; wash/liquidity are proxies; spoofing/layering out of scope; modest per-split event counts; independent BTC/ETH events.')
print('    Contract/task discrepancies recorded:', len(DISCREPANCIES))
print('15. Training-readiness verdict:', VERDICT)
print('=========================================================================')
if not ALL_PASS: raise AssertionError('Layer 3 v2 validation failed; see the FAIL lines above (nothing was published).')

PUBLISHED -> /content/drive/MyDrive/research datasets/processed_layer3_final_v2

==================== LAYER 3 V2 — FINAL SHARE BLOCK ====================
1. Output directory: /content/drive/MyDrive/research datasets/processed_layer3_final_v2
2. Files created: 32 (btc/eth x train/val/test x features/labels/state, 2 event tables, metadata, validation, reports)
3. Rows per symbol: {'BTCUSDT': {'train': 90720, 'val': 18720, 'test': 20160}, 'ETHUSDT': {'train': 90720, 'val': 18720, 'test': 20160}}
4. Scenario families: ['pump_and_dump_like', 'fake_breakout_like', 'abnormal_volume_burst', 'trade_flow_imbalance', 'wash_like_aggregate_proxy', 'liquidity_stress_proxy', 'composite'] | compatible composite pairs: [('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('D', 'C'), ('F', 'C'), ('F', 'D')]
5. Event counts (symbol, split, family):
scenario_family  abnormal_volume_burst  composite  fake_breakout_like  liquidity_stress_proxy  pump_and_dump_like  trade_flow_imbalance  wash_like_aggregate_proxy